# 38 — Project Extraction
**Goal:** Extract project names, tech stacks, and outcomes from resumes.

Projects are where candidates show initiative — side builds, capstones, open-source work — often formatted like mini-experience entries but without dates. This chapter parses the typical "Name | Tech Stack" header plus bullets, reusing the state-machine pattern from Ch. 36 with a simpler schema.

**Why it matters for resumes / ATS:** for junior candidates, projects *are* the experience. Extracting the name and tech stack lets an ATS match project tech against JD requirements, and the outcome bullets ("Achieved 92% accuracy…", "99.9% uptime") carry the same quantified-impact signal Ch. 37 scores — giving candidates without formal work history a fair shot.

## 1. Project Pattern Recognition

Project blocks follow their own template: a header line `Name | Tech Stack` (pipe-delimited), followed by bullets that describe what was built and what it achieved. Note what is missing versus experience: no company, no dates.

**What the code does:** sets up `project_text` with three sample projects — "Resume Intelligence Platform | Python, NLP, TensorFlow", "Sentiment Analysis Dashboard | Python, Flask, React", and "E-commerce Recommendation Engine | Python, Spark, MongoDB" — and prints the expected shape: name, tech stack, bullets with outcomes.

**Why it matters:** the pipe delimiter is the parser's anchor — it is rare in ordinary prose, so a line containing `|` is almost certainly a project header. That single observation keeps the parser nearly as simple as the education extractor.

In [ ]:
project_text = """PROJECTS
Resume Intelligence Platform | Python, NLP, TensorFlow
- Built end-to-end resume parsing and ATS scoring system
- Achieved 92% accuracy on skill extraction

Sentiment Analysis Dashboard | Python, Flask, React
- Real-time sentiment analysis for 10K+ tweets/day
- Deployed on AWS with 99.9% uptime

E-commerce Recommendation Engine | Python, Spark, MongoDB
- Collaborative filtering for 1M+ users
- Increased conversion rate by 25%
"""
print("Projects often have: Name, Tech Stack, Bullets with outcomes")

## 2. Project Parser

`extract_projects()` is the Ch. 36 state machine, simplified: a header regex captures the name and tech stack, and every following non-empty line becomes a bullet of the current project.

**What the code does:** the header pattern `^([A-Za-z\s]+)\s*[|]\s*(.+)$` splits on the first pipe; anything else is bullet-stripped (leading `-`/`•`/whitespace removed) and appended to `current["bullets"]`; a new header finalizes the previous project.

**Verified on the sample:** the first project parses cleanly — "Resume Intelligence Platform" with 2 bullets. The second, "Sentiment Analysis Dashboard", ends up with **5 bullets** because the third project's header line — "E-commerce Recommendation Engine | Python, Spark, MongoDB" — fails to match: the hyphen in "E-commerce" is not in `[A-Za-z\s]+`, so the line is treated as a bullet and drags its two outcome bullets along. A textbook case of a delimiter assumption failing on real data.

**Try it:** add `-` to the header character class (e.g. `[A-Za-z\s-]+`) and all three projects parse — a one-character fix with a visible payoff.

In [ ]:
import re

def extract_projects(text):
    projects = []
    lines = text.split("\n")
    current = None
    for line in lines:
        ls = line.strip()
        if not ls: continue
        # Project header: Name | Tech Stack
        proj_match = re.match(r"^([A-Za-z\s]+)\s*[|]\s*(.+)$", ls)
        if proj_match:
            if current: projects.append(current)
            current = {"name": proj_match.group(1).strip(), "tech": proj_match.group(2).strip(), "bullets": []}
            continue
        bullet = re.sub(r"^[\s•\-*–]+", "", ls)
        if bullet and current:
            current["bullets"].append(bullet)
    if current: projects.append(current)
    return projects

for p in extract_projects(project_text):
    print(f"\n  {p['name']:35s} | {p['tech']}")
    for b in p['bullets']:
        print(f"    - {b}")

## Summary: Project extraction follows similar pattern to experience but without dates.

**Projects reuse the experience state machine, minus dates, plus a pipe-delimited header.**

One regex over the header line and a bullet strip are enough to recover name, tech stack, and outcome bullets from a standard projects block — and the hyphen-in-name failure above is the real lesson: every parser encodes assumptions about delimiters, and those assumptions fail on real data in predictable ways. Validate against a corpus before trusting recall.

Project dicts (`name`, `tech`, `bullets`) slot into Ch. 39's `projects` list, and their outcome bullets are scored by the same Ch. 37 STAR logic. With all four content sections parsed, Ch. 39 finally assembles everything into one schema.